# Pareto Front Analysis — 3 Objectives

**Objectives:**
- Maximize **X**
- Minimize **Y**
- Minimize **Z**

**Parameters:** A, B, C, D, E, F

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from itertools import combinations

## 1. Load data

In [ ]:
# ── Edit this path ──────────────────────────────────────────────────────────
CSV_PATH = 'your_data.csv'
# ────────────────────────────────────────────────────────────────────────────

PARAMS = ['A', 'B', 'C', 'D', 'E', 'F']
OBJ_MAXIMIZE = ['X']          # higher is better
OBJ_MINIMIZE = ['Y', 'Z']     # lower is better
ALL_OBJ = OBJ_MAXIMIZE + OBJ_MINIMIZE

df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} rows, columns: {list(df.columns)}')
df.head()

## 2. Pareto front — n objectives, mixed directions

In [ ]:
def pareto_front(df, objectives, directions):
    """
    Return a boolean mask of Pareto-optimal rows.

    Parameters
    ----------
    df         : DataFrame
    objectives : list of column names
    directions : list of 'max' or 'min', one per objective

    Returns
    -------
    np.ndarray of bool, True where the row is Pareto-optimal
    """
    # Convert everything to a 'higher is better' matrix for easy comparison
    vals = df[objectives].values.astype(float)
    sign = np.array([1 if d == 'max' else -1 for d in directions], dtype=float)
    M = vals * sign          # shape (n_rows, n_obj)

    n = len(M)
    dominated = np.zeros(n, dtype=bool)

    for i in range(n):
        if dominated[i]:
            continue
        # A row j dominates row i when:
        #   j is >= i in every objective  AND  strictly > in at least one
        diff = M - M[i]          # (n, n_obj): positive means j beats i
        all_ge  = (diff >= 0).all(axis=1)   # j no worse than i everywhere
        any_gt  = (diff >  0).any(axis=1)   # j strictly better somewhere
        dominators = all_ge & any_gt
        dominators[i] = False               # can't dominate yourself
        if dominators.any():
            dominated[i] = True

    return ~dominated


directions = ['max' if o in OBJ_MAXIMIZE else 'min' for o in ALL_OBJ]

df['pareto'] = pareto_front(df, ALL_OBJ, directions)
pareto_df = df[df['pareto']].sort_values('X', ascending=False)

print(f'Pareto-optimal experiments: {len(pareto_df)} / {len(df)}')
pareto_df[PARAMS + ALL_OBJ].head(10)

## 3. Pairwise 2-D projections of the Pareto front

In [ ]:
pairs = list(combinations(ALL_OBJ, 2))   # (X,Y), (X,Z), (Y,Z)
fig, axes = plt.subplots(1, len(pairs), figsize=(5 * len(pairs), 4))

for ax, (ox, oy) in zip(axes, pairs):
    ax.scatter(df[ox],        df[oy],        alpha=0.25, s=15, label='All')
    ax.scatter(pareto_df[ox], pareto_df[oy], color='red', s=40,
               zorder=5, label='Pareto front')

    dir_x = '(max)' if ox in OBJ_MAXIMIZE else '(min)'
    dir_y = '(max)' if oy in OBJ_MAXIMIZE else '(min)'
    ax.set_xlabel(f'{ox} {dir_x}')
    ax.set_ylabel(f'{oy} {dir_y}')
    ax.set_title(f'{ox} vs {oy}')
    ax.legend()

plt.suptitle('Pareto front — pairwise projections', fontsize=13)
plt.tight_layout()
plt.show()

## 4. 3-D scatter of all three objectives

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')

rest = df[~df['pareto']]
ax.scatter(rest['X'],       rest['Y'],       rest['Z'],
           alpha=0.2, s=10, label='All experiments', c='steelblue')
ax.scatter(pareto_df['X'], pareto_df['Y'], pareto_df['Z'],
           color='red', s=50, zorder=5, label='Pareto front')

ax.set_xlabel('X (max)')
ax.set_ylabel('Y (min)')
ax.set_zlabel('Z (min)')
ax.set_title('3-D Pareto front')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Parallel-coordinates plot of Pareto-optimal experiments

In [ ]:
from pandas.plotting import parallel_coordinates

plot_cols = PARAMS + ALL_OBJ
plot_df = df[plot_cols + ['pareto']].copy()

# Normalise each column to [0, 1] for visual clarity
norm = plot_df[plot_cols].copy()
norm = (norm - norm.min()) / (norm.max() - norm.min() + 1e-12)
norm['group'] = plot_df['pareto'].map({True: 'Pareto', False: 'Other'})

fig, ax = plt.subplots(figsize=(12, 4))
parallel_coordinates(norm, class_column='group',
                     color=['red', 'steelblue'], alpha=0.4, ax=ax)
ax.set_title('Parallel coordinates — normalised parameters & objectives')
ax.set_xticklabels(plot_cols, rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 6. Export Pareto-optimal rows

In [ ]:
out_path = CSV_PATH.replace('.csv', '_pareto.csv')
pareto_df[PARAMS + ALL_OBJ].to_csv(out_path, index=False)
print(f'Saved {len(pareto_df)} Pareto-optimal rows to {out_path}')